# V4.1 — 03: Step 2 Training (Category & Sentiment)


## Bootstrap
Mendefinisikan ulang `step_stage`, `require_vars`, dan path modul yang tidak tersimpan di `pipeline_state.pkl`.

In [ ]:
# ============================================================
#  Bootstrap: definisi runtime yang tidak tersimpan di pipeline_state.pkl
#  Dijalankan otomatis sebelum state recovery di setiap notebook serial.
# ============================================================
import os, sys, time, json, re, pickle, shutil, glob, importlib, warnings
from datetime import datetime

# --- step_stage & require_vars (dari sel 6 V4.1) ---
class step_stage:
    def __init__(self, title, total_steps=None):
        self.title = title
        self.total = total_steps
        self.n = 0
        self.t0 = None
    def __enter__(self):
        self.t0 = time.time()
        print("=" * 78)
        print(f"\u25b6\ufe0f  {self.title}")
        print("=" * 78, flush=True)
        return self
    def step(self, msg):
        self.n += 1
        tag = f"{self.n}/{self.total}" if self.total else str(self.n)
        print(f"   [{tag}] {time.time() - self.t0:6.1f}s  {msg}", flush=True)
    def note(self, msg):
        print(f"        {msg}", flush=True)
    def __exit__(self, exc_type, exc, tb):
        dur = time.time() - self.t0
        if exc_type is None:
            print(f"\u2705 {self.title} \u2014 selesai dalam {dur:.1f}s\n", flush=True)
        else:
            print(f"\u274c {self.title} \u2014 gagal setelah {dur:.1f}s: {exc}\n", flush=True)
        return False

def require_vars(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Variabel {missing} belum ada di memori. Jalankan sel pemulihan state "
            f"(6b/6c) atau notebook 01_setup lebih dulu.")

def write_stage_progress(path, **fields):
    d = {"saved_at": datetime.now().isoformat()}
    d.update(fields)
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                old = json.load(f)
            if isinstance(old, dict):
                d["previous"] = old
        except Exception:
            pass
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1

# --- patch_eval_metrics_counts (ringkas; full patch dijalankan ulang di sel 5a/8a) ---
# Pastikan nama fungsi tersedia agar require_vars tidak gagal bila sel import
# upstream belum berjalan. Patch sebenarnya dilakukan di sel step1/step2 init.
def patch_eval_metrics_counts():
    try:
        import eval_metrics as _em
    except ImportError:
        return "eval_metrics belum di-import (akan dipatch di sel init)"
    return "eval_metrics siap"

def history_display_frame(history, epochs_col="epoch"):
    import pandas as pd
    if not history:
        return pd.DataFrame()
    return pd.DataFrame(history)

def metrics_display_frame(res):
    import pandas as pd
    if not res:
        return pd.DataFrame()
    rows = [{"Metric": k, "Value": v} for k, v in res.items()]
    return pd.DataFrame(rows)

def best_epoch_row(history, f1_key="micro-F1"):
    if not history:
        return None, 0.0, None
    best = max(history, key=lambda r: float(r.get(f1_key, 0.0)))
    return best, float(best.get(f1_key, 0.0)), int(best.get("epoch", 0))

def unpack_model_output(out):
    losses, logits = out
    loss = losses[0] if isinstance(losses, (list, tuple)) else losses
    return loss, logits

# --- Path setup (dari sel 8 V4.1, versi ringkas) ---
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if "base_project_dir" not in globals() or not globals().get("base_project_dir"):
    if os.path.exists("/content/drive/MyDrive/ACOS"):
        base_project_dir = "/content/drive/MyDrive/ACOS"
    elif os.path.exists("/content/ACOS"):
        base_project_dir = "/content/ACOS"
    else:
        base_project_dir = os.path.abspath(".")
    extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
    data_root = os.path.join(base_project_dir, "data")

# sys.path: upstream ACOS + acos_id
for _p in [extract_dir, os.path.join(extract_dir, "absa5")]:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

# --- acos_id reimport ---
def _cari_indo_root():
    for _d in ["/content/drive/MyDrive/ACOS-IndoBERT",
               "/content/drive/MyDrive/ACOS/ACOS-IndoBERT",
               "/content/drive/MyDrive/ACOS-ASLI/ACOS-IndoBERT",
               os.path.join(base_project_dir, "ACOS-IndoBERT"),
               os.path.abspath("ACOS-IndoBERT"),
               os.path.abspath(os.path.join("..", "ACOS-IndoBERT"))]:
        if os.path.isdir(os.path.join(_d, "acos_id")):
            return _d
    return None

indo_root = globals().get("indo_root") or _cari_indo_root()
if indo_root and indo_root not in sys.path:
    sys.path.insert(0, indo_root)
if indo_root:
    try:
        acos_id = importlib.import_module("acos_id")
        acos_taxonomy = importlib.import_module("acos_id.taxonomy")
        acos_selftest = importlib.import_module("acos_id.selftest")
        acos_ckpt = importlib.import_module("acos_id.checkpoint")
        acos_eda = importlib.import_module("acos_id.eda")
        acos_upstream = importlib.import_module("acos_id.upstream")
    except ModuleNotFoundError:
        pass  # akan dipatch di sel init masing-masing step

# _backbone_dirname untuk recovery state
BACKBONE_DIRNAME = {
    "indobert": "indobert_base_p1",
    "indobert-large": "indobert_large_p1",
    "bert-en": "bert_base_uncased",
}
def _backbone_dirname(backbone=None):
    key = backbone or globals().get("BACKBONE") or "bert-en"
    return BACKBONE_DIRNAME.get(key, str(key).replace("-", "_"))

print(f"\u26a1 Bootstrap siap | base_project_dir={base_project_dir} | indo_root={indo_root}")


## State Recovery
Memuat `pipeline_state.pkl` dari sesi sebelumnya. Jika belum ada, jalankan notebook `01_setup` lebih dulu.

### 6b. Smart State Recovery (Gunakan Sel Ini Jika Kernel Reconnect / Restart)

In [ ]:
# Sel Pemulihan Cerdas: Otomatis mendeteksi sesi aktif terakhir.
# Memulihkan BUKAN hanya config/path, tapi juga seluruh artefak runtime yang tersimpan.
def auto_find_latest_state(search_bases, domain="rest16"):
    """Mencari berkas state pipeline_state.pkl dengan validasi domain."""
    if isinstance(search_bases, str):
        search_bases = [search_bases]

    # 1. Cek pointer langsung
    for sb in search_bases:
        if not sb or not os.path.isdir(sb):
            continue
        pointer = os.path.join(sb, f"latest_pipeline_state_{domain}.pkl")
        if os.path.exists(pointer):
            return pointer

    # 2. Cari mendalam
    candidates = []
    for sb in search_bases:
        if not sb or not os.path.exists(sb):
            continue
        for root, dirs, files in os.walk(sb):
            if "pipeline_state.pkl" in files:
                p = os.path.join(root, "pipeline_state.pkl")
                norm = p.replace(os.sep, "/")
                if domain and f"/{domain}_" not in norm and f"_{domain}/" not in norm and f"/{domain}/" not in norm:
                    try:
                        with open(p, "rb") as f:
                            s = pickle.load(f)
                        if s.get("DOMAIN") != domain:
                            continue
                    except Exception:
                        continue
                candidates.append((os.path.getmtime(p), p))
    if candidates:
        candidates.sort(reverse=True)
        return candidates[0][1]
    return None


## 7. Jembatan Pasangan Kandidat (Step 1 → Step 2)

### 7a. Pembentukan / Pemuatan Pasangan
Membaca `pred4pipeline.txt` dan membentuk cross-product aspect × opinion. Tag dikenali
dari polanya, bukan posisi kolom tab, supaya tag seperti `a--1,-1` tidak menyelundup ke
kolom teks dan memicu KeyError saat tokenisasi Step 2.

In [ ]:
ensure_objects()
require_vars("step_stage", "session_dirs", "extract_dir")

import codecs as cs
import re

with step_stage("7a. Pasangan kandidat Step 1 → Step 2", 4) as st:
    pred_file = os.path.join(session_dirs["logs"], "pred4pipeline.txt")
    target_tokenized_tsv = os.path.join(tokenized_base, "tokenized_data",
                                        f"{DOMAIN}_test_pair_1st.tsv")
    candidate_csv = os.path.join(session_dirs["csv"], "candidate_pairs_summary.csv")

    _in_mem = ('df_pairs' in globals() and df_pairs is not None
               and not df_pairs.empty and os.path.exists(target_tokenized_tsv))
    _on_disk = os.path.exists(candidate_csv) and os.path.exists(target_tokenized_tsv)

    if _in_mem:
        st.step(f"[CACHE HIT] {len(df_pairs):,} pasangan sudah ada di memori runtime")
    elif _on_disk:
        df_pairs = pd.read_csv(candidate_csv)
        st.step(f"[CACHE HIT] {len(df_pairs):,} pasangan dimuat dari {candidate_csv}")
    else:
        if not os.path.exists(pred_file):
            found_pred = auto_find_file("pred4pipeline.txt")
            if found_pred:
                os.makedirs(session_dirs["logs"], exist_ok=True)
                shutil.copy(found_pred, pred_file)
                st.step(f"pred4pipeline.txt disalin dari sesi sebelumnya: {found_pred}")
            else:
                raise FileNotFoundError(
                    f"pred4pipeline.txt tidak ada di {pred_file} maupun sesi lain. "
                    f"Jalankan Step 1 (sel 5a-5f) lebih dulu.")
        else:
            st.step(f"Sumber prediksi: {pred_file}")

        with cs.open(pred_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        st.step(f"{len(lines):,} baris prediksi dibaca")

        TAG_RE = re.compile(r'^(a|o)-(-?\d+,-?\d+)$')
        pair_records = []
        n_skip = 0
        os.makedirs(os.path.dirname(target_tokenized_tsv), exist_ok=True)
        with cs.open(target_tokenized_tsv, 'w', encoding='utf-8') as wf:
            for line in tqdm(lines, desc="   Membentuk pasangan", unit="baris", leave=False):
                asp, opi, text_parts = [], [], []
                for tok in line.strip().split():
                    m = TAG_RE.match(tok)
                    if m:
                        (asp if m.group(1) == 'a' else opi).append(m.group(2))
                    else:
                        text_parts.append(tok)
                if not text_parts:
                    n_skip += 1
                    continue
                text = ' '.join(text_parts)
                if not asp:
                    asp.append('-1,-1')
                if not opi:
                    opi.append('-1,-1')
                for pa in asp:
                    for po in opi:
                        wf.write(f"{text}####{pa} {po}\n")
                        pair_records.append({"Text": text, "Aspect_Span": pa,
                                             "Opinion_Span": po})

        df_pairs = pd.DataFrame(pair_records)
        df_pairs.to_csv(candidate_csv, index=False)
        st.step(f"{len(df_pairs):,} pasangan ditulis ke {target_tokenized_tsv}"
                + (f" ({n_skip} baris kosong dilewati)" if n_skip else ""))

    st.step(f"Siap untuk Step 2 | berkas pair: "
            f"{'ada' if os.path.exists(target_tokenized_tsv) else 'BELUM ADA'}")

### 7b. Distribusi Tipe Pasangan
Sel pelaporan: tabel implicit/explicit, plot batang, dan penyimpanan state. Aman diulang.

In [ ]:
require_vars("step_stage", "df_pairs")

with step_stage("7b. Laporan distribusi pasangan kandidat", 4) as st:
    rep.section("4. Jembatan: pasangan kandidat")
    if df_pairs.empty:
        rep.text("Tidak ada pasangan kandidat yang terbentuk.")
        st.step("df_pairs kosong — tabel dan plot dilewati")
    else:
        df_pairs["Is_Implicit_Aspect"] = df_pairs["Aspect_Span"] == "-1,-1"
        df_pairs["Is_Implicit_Opinion"] = df_pairs["Opinion_Span"] == "-1,-1"
        df_pairs["Pair_Type"] = (
            df_pairs["Is_Implicit_Aspect"].map({True: "Implicit", False: "Explicit"}) + "-"
            + df_pairs["Is_Implicit_Opinion"].map({True: "Implicit", False: "Explicit"})
        )
        n_pair = len(df_pairs)
        df_tipe = df_pairs["Pair_Type"].value_counts().rename_axis(
            "Tipe_Pasangan").reset_index(name="Jumlah")
        df_tipe["Persen"] = (df_tipe["Jumlah"] / n_pair * 100).round(2)
        st.step(f"{n_pair:,} pasangan dalam {len(df_tipe)} tipe: "
                + ", ".join(f"{r.Tipe_Pasangan} {r.Persen}%" for r in df_tipe.itertuples()))

        export_step_table(df_tipe, name="master_04_tipe_pasangan", csv_dir=csv_dir,
                          md_dir=md_dir,
                          title=f"Distribusi Tipe Pasangan Kandidat ({DOMAIN.upper()})",
                          notes=f"Total {n_pair} pasangan dari cross-product aspect x opinion.")
        rep.table(df_tipe, caption="Tipe pasangan")
        export_step_table(df_pairs.head(20), name="master_05_preview_pasangan",
                          csv_dir=csv_dir, md_dir=md_dir,
                          title=f"Preview 20 Pasangan Kandidat ({DOMAIN.upper()})",
                          max_rows_md=20)
        st.step("Tabel master_04 & master_05 diekspor")

        plt.figure(figsize=(9, 5))
        _w = ["#3498db", "#9b59b6", "#e67e22", "#e74c3c"][:len(df_tipe)]
        _b = plt.bar(df_tipe["Tipe_Pasangan"], df_tipe["Jumlah"], color=_w,
                     edgecolor="black", alpha=0.88)
        for b, v in zip(_b, df_tipe["Jumlah"]):
            plt.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / n_pair * 100:.1f}%)",
                     ha="center", va="bottom", fontsize=9, fontweight="bold")
        plt.title(f"[{DOMAIN.upper()}] Pasangan Kandidat Step 1 -> Step 2",
                  fontsize=12, fontweight="bold")
        plt.ylabel("Jumlah pasangan")
        plt.margins(y=0.18)
        plt.grid(axis="y", linestyle="--", alpha=0.7)
        plt.tight_layout()
        _pp = os.path.join(plots_dir, "04_candidate_pairs_distribution.png")
        plt.savefig(_pp, dpi=300)
        plt.show()
        plt.close()
        rep.image(_pp, "Distribusi tipe pasangan kandidat")
        st.step(f"Plot disimpan: {_pp}")

    update_mcp_manifest("CANDIDATE_PAIRS_GENERATED", 4,
                        {"candidate_pairs_count": len(df_pairs)})
    save_pipeline_state({"df_pairs": df_pairs})
    st.step("Manifest → CANDIDATE_PAIRS_GENERATED, pipeline_state.pkl diperbarui")

## 8. Step 2: Klasifikasi Category & Sentiment (Bertahap)
Melatih `CategorySentiClassification` multi-label pada pasangan kandidat $(a, o)$.
Dipecah mengikuti pola Step 1 supaya setiap bagian bisa dilacak sendiri.

| Sel | Isi | Aman diulang |
|---|---|---|
| 8a | Import, patch tokenizer, label, path checkpoint | ya |
| 8b | Deteksi cache Step 2 (sesi aktif + sesi lama) | ya |
| 8c | Data evaluasi pasangan + gold Step 2 | ya |
| 8d | Model, data training, optimizer | ya (mengalokasi ulang VRAM) |
| 8e | Loop training per epoch + checkpoint terbaik | tidak (melatih ulang) |
| 8f | Plot, tabel laporan, manifest, simpan state | ya |

Sel 8c-8e melewati dirinya sendiri saat `STEP2_SKIP_TRAINING` bernilai `True`.
Catatan: 8c tetap dijalankan meski cache hit bila `eval_loader_2` belum ada, karena
sel evaluasi final (9a) membutuhkannya.

### 8a. Inisialisasi Step 2

In [ ]:
ensure_objects()
require_vars("step_stage", "session_dirs", "bert_cache_dir")

from modeling import CategorySentiClassification
from modeling import BertModel, BertPreTrainedModel
import torch.nn as nn
class CategorySentiDualHead(BertPreTrainedModel):
    """Step 2 Dual-Head: representasi kandidat pasangan (aspect-opinion) bersama,
    dengan dua output head terpisah:
    - category_head  : Linear(hidden_size * 2, num_categories) — multi-label BCE
    - sentiment_head : Linear(hidden_size * 2, num_sentiments) — multi-class CE

    Menghasilkan kombinasi fused_logits (num_categories * num_sentiments) untuk
    kompatibilitas 100% dengan `pair_eval` upstream, sekaligus mengekspos
    `cat_logits` dan `senti_logits` untuk evaluasi mandiri.
    """
    def __init__(self, config, num_categories=13, num_sentiments=3,
                 output_attentions=False, keep_multihead_output=False, **kwargs):
        super(CategorySentiDualHead, self).__init__(config)
        self.output_attentions = output_attentions
        self.num_categories = num_categories
        self.num_sentiments = num_sentiments
        self.num_labels = [num_categories * num_sentiments, 2]
        self.bert = BertModel(config, output_attentions=output_attentions,
                              keep_multihead_output=keep_multihead_output)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.category_head = nn.Linear(config.hidden_size * 2, num_categories)
        self.sentiment_head = nn.Linear(config.hidden_size * 2, num_sentiments)
        self.apply(self.init_bert_weights)

    def forward(self, tokenizer, _e, aspect_input_ids,
                aspect_token_type_ids, aspect_attention_mask,
                candidate_aspect, candidate_opinion, label_id=None):
        aspect_seq_len = torch.max(torch.sum(aspect_attention_mask, dim=-1))
        max_seq_len = aspect_seq_len
        aspect_input_ids = aspect_input_ids[:, :max_seq_len].contiguous()
        aspect_token_type_ids = aspect_token_type_ids[:, :max_seq_len].contiguous()
        aspect_attention_mask = aspect_attention_mask[:, :max_seq_len].contiguous()
        candidate_aspect = candidate_aspect[:, :max_seq_len].contiguous()
        candidate_opinion = candidate_opinion[:, :max_seq_len].contiguous()

        pooled_outputs, pooled_output = self.bert(
            aspect_input_ids, aspect_token_type_ids, aspect_attention_mask,
            output_all_encoded_layers=False, head_mask=None
        )

        hidden_size = pooled_output.shape[-1]

        candidate_aspect_sum = torch.sum(candidate_aspect, -1).float()
        aspect_denominator = (candidate_aspect_sum + candidate_aspect_sum.eq(0).float()).unsqueeze(-1).repeat(1, hidden_size)
        candidate_aspect_rep = torch.div(
            torch.matmul(candidate_aspect.float().unsqueeze(1), pooled_outputs).squeeze(1),
            aspect_denominator
        )

        candidate_opinion_sum = torch.sum(candidate_opinion, -1).float()
        opinion_denominator = (candidate_opinion_sum + candidate_opinion_sum.eq(0).float()).unsqueeze(-1).repeat(1, hidden_size)
        candidate_opinion_rep = torch.div(
            torch.matmul(candidate_opinion.float().unsqueeze(1), pooled_outputs).squeeze(1),
            opinion_denominator
        )

        rep = self.dropout(torch.cat([candidate_aspect_rep, candidate_opinion_rep], -1))
        cat_logits = self.category_head(rep)       # (batch, num_categories)
        senti_logits = self.sentiment_head(rep)    # (batch, num_sentiments)

        loss = None
        if label_id is not None:
            reshaped = label_id.view(-1, self.num_categories, self.num_sentiments)
            cat_targets = reshaped.sum(dim=-1).clamp(max=1.0)  # multi-hot (batch, 13)

            senti_per_sample = reshaped.sum(dim=1)  # (batch, 3)
            has_senti = (senti_per_sample.sum(dim=-1) > 0)
            senti_targets = torch.where(
                has_senti,
                senti_per_sample.argmax(dim=-1),
                torch.full((label_id.size(0),), -1, dtype=torch.long, device=label_id.device)
            )

            bce = nn.BCEWithLogitsLoss()
            cat_loss = bce(cat_logits, cat_targets.float())
            ce = nn.CrossEntropyLoss(ignore_index=-1)
            senti_loss = ce(senti_logits, senti_targets)
            loss = cat_loss + senti_loss

        # Rekonstruksi fused_logits untuk pair_eval upstream: (batch, num_categories * num_sentiments)
        # Prediksi pasangan (c, s) aktif jika kategori c aktif (cat_logits[c] > 0)
        # dan s adalah argmax dari head sentimen
        bs = cat_logits.size(0)
        best_senti = senti_logits.argmax(dim=-1)
        expanded_cat = cat_logits.unsqueeze(2).expand(-1, -1, self.num_sentiments)
        senti_mask = (torch.arange(self.num_sentiments, device=cat_logits.device).unsqueeze(0).unsqueeze(0) == best_senti.unsqueeze(1).unsqueeze(2))
        fused_logits = torch.where(senti_mask, expanded_cat, expanded_cat - 10000.0).view(bs, -1)

        self.latest_cat_logits = cat_logits.detach()
        self.latest_senti_logits = senti_logits.detach()

        if loss is not None:
            return [loss], [fused_logits]
        return [fused_logits]

from dataset_utils import read_pair_gold
from eval_metrics import pair_eval
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from bert_utils.tokenization import BertTokenizer
from bert_utils.optimization import BertAdam
from run_classifier_dataset_utils import processors, output_modes
from tqdm.auto import tqdm
import logging

# Toggle Melatih Ulang Step 2 (Set True jika ingin memaksa melatih ulang)
FORCE_RETRAIN_STEP2 = False

with step_stage("8a. Inisialisasi Step 2: patch tokenizer, patch metrik, label, path", 6) as st:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        st.step(f"GPU cache dibersihkan | {torch.cuda.get_device_name(0)}")
    else:
        st.step("Mode CPU aktif (CUDA tidak tersedia)")

    # Token di luar vocab dipetakan ke [UNK] dan dilaporkan, bukan melempar
    # KeyError yang menghentikan seluruh epoch.
    _oov_seen = set()

    def patched_convert_tokens_to_ids(self, tokens):
        if tokens is None:
            return None
        if isinstance(tokens, str):
            try:
                return self.vocab[tokens]
            except KeyError:
                if tokens not in _oov_seen:
                    _oov_seen.add(tokens)
                    print(f"⚠️ Token di luar vocab: {ascii(tokens)}")
                return self.vocab.get('[UNK]', 100)
        ids = []
        for token in tokens:
            try:
                ids.append(self.vocab[token])
            except KeyError:
                _oov_seen.add(token)
                ids.append(self.vocab.get('[UNK]', 100))
        if len(ids) > self.max_len:
            logging.getLogger(__name__).warning(
                f"Seq len ({len(ids)}) > max ({self.max_len})")
        return ids

    BertTokenizer.convert_tokens_to_ids = patched_convert_tokens_to_ids
    st.step("BertTokenizer.convert_tokens_to_ids dipatch (OOV → [UNK], dicatat sekali)")

    # Patch evaluasi: measureQuad & measureQuad_imp ikut mengembalikan tp/fp/fn,
    # sekaligus defensif terhadap teks prediksi di luar gold (KeyError text_type)
    # dan memperbaiki return-di-luar-loop pada measureQuad_imp.
    _em2 = patch_eval_metrics_counts()
    st.step("eval_metrics dipatch: tp/fp/fn ikut dikembalikan, agregat semua slot "
            "difficulty, aman terhadap OOV/mismatched text")

    processor_step2 = processors["categorysenti"]()
    label_list_step2 = processor_step2.get_labels(DOMAIN)
    num_labels_step2 = len(label_list_step2[0])
    st.step(f"Label category-sentiment: {num_labels_step2} kelas")

    # Dual-Head: label terpisah per komponen (hanya domain Indonesia)
    USE_DUAL_HEAD = acos_taxonomy.is_id_domain(DOMAIN)
    if USE_DUAL_HEAD:
        if hasattr(acos_taxonomy, "patch_processor_labels_dualhead"):
            _dh_patch = acos_taxonomy.patch_processor_labels_dualhead(processors)
        else:
            # Fallback jika acos_taxonomy di storage/Drive belum memiliki patch dual-head
            cs_cls = processors["categorysenti"]
            _cats = getattr(acos_taxonomy, "CATEGORIES", [
                "ONBOARDING_KYC", "AUTH_ACCESS", "TRANSACTION_TRANSFER",
                "APP_PERFORMANCE", "UI_UX_DESIGN", "FEES_CHARGES",
                "INTEREST_RETURNS", "CUSTOMER_SERVICE", "SECURITY_FRAUD",
                "FEATURES_PRODUCT", "PROMO_MARKETING", "NOTIFICATION_INFO",
                "ACCOUNT_MANAGEMENT"
            ])
            cs_cls.get_labels_category = lambda self, dt: list(_cats) if acos_taxonomy.is_id_domain(dt) else None
            cs_cls.get_labels_sentiment = lambda self, dt: ["0", "1", "2"] if acos_taxonomy.is_id_domain(dt) else None
            cs_cls._acos_id_dualhead_patched = True
            _dh_patch = {"patched": True, "fallback": True, "num_labels_category": len(_cats), "num_labels_sentiment": 3}
        label_list_cat   = processor_step2.get_labels_category(DOMAIN)
        label_list_senti = processor_step2.get_labels_sentiment(DOMAIN)
        num_labels_cat   = len(label_list_cat)   # 13
        num_labels_senti = len(label_list_senti)  # 3
        st.step(f"Dual-Head aktif: {num_labels_cat} kategori | "
                f"{num_labels_senti} sentimen (domain Indonesia)")
    else:
        USE_DUAL_HEAD = False
        st.step(f"Single-Head: {num_labels_step2} label gabungan "
                f"(domain Inggris {DOMAIN})")

    step2_ckpt = session_dirs["step2_checkpoint"]
    step2_bin = os.path.join(step2_ckpt, "pytorch_model.bin")
    step2_csv = os.path.join(session_dirs["csv"], "step2_training_history.csv")
    step2_progress_json = os.path.join(session_dirs["logs"], "step2_progress.json")
    os.makedirs(step2_ckpt, exist_ok=True)
    st.step(f"Checkpoint : {step2_ckpt}")

    # args_h dan logger2 dibangun di sini, bukan di 8d, karena 8d dilewati saat
    # cache hit sementara sel 8e dan 9a tetap membutuhkannya.
    logger2 = logging.getLogger("Step2")
    if "args_h" not in globals() or globals()["args_h"] is None:
        import types as _t
        args_h = _t.SimpleNamespace(output_dir=session_dirs["logs"],
                                    max_seq_length=MAX_SEQ_LENGTH)
        st.step(f"args_h dibangun (Step 1 dilewati di sesi ini) → {args_h.output_dir}")
    else:
        st.step(f"args_h dipakai ulang dari Step 1 → {args_h.output_dir}")

    print(f"   FORCE_RETRAIN_STEP2={FORCE_RETRAIN_STEP2} | epoch target={NUM_EPOCHS} | "
          f"batch={STEP2_BATCH_SIZE} | lr={STEP2_LR}")

### 8b. Deteksi Cache Step 2
Menentukan `STEP2_SKIP_TRAINING`, satu-satunya penentu apakah sel 8d-8e melatih model.

In [ ]:
require_vars("step_stage", "step2_bin", "step2_csv", "FORCE_RETRAIN_STEP2")

with step_stage("8b. Deteksi cache Step 2 (sesi aktif lalu sesi lama)", 4) as st:
    step2_already_done = os.path.exists(step2_bin)
    st.step("Sesi aktif — model: " + (
        f"{os.path.getsize(step2_bin) / 1024 ** 2:.1f} MB" if step2_already_done
        else "belum ada"))

    if step2_already_done:
        st.step("Pencarian sesi lama dilewati (checkpoint sesi aktif sudah ada)")
    else:
        found_bin2 = auto_find_file("pytorch_model.bin", must_contain="step2_best",
                                    search_roots=[
                                        results_base if 'results_base' in globals() else "",
                                        "/content/drive/MyDrive/ACOS/Output/results",
                                        os.path.join(base_project_dir, "Output", "results"),
                                    ])
        if found_bin2 and "step2_best" in found_bin2:
            src_dir2 = os.path.dirname(found_bin2)
            st.step(f"Checkpoint sesi sebelumnya ditemukan: {src_dir2}")
            for fn in ["pytorch_model.bin", "config.json", "vocab.txt"]:
                fp = os.path.join(src_dir2, fn)
                if os.path.exists(fp):
                    shutil.copy(fp, os.path.join(step2_ckpt, fn))
                    st.note(f"↪ {fn} ({os.path.getsize(fp) / 1024 ** 2:.1f} MB) disalin")
            found_csv2 = auto_find_file("step2_training_history.csv")
            if found_csv2:
                shutil.copy(found_csv2, step2_csv)
                st.note(f"↪ step2_training_history.csv disalin dari {found_csv2}")
            step2_already_done = os.path.exists(step2_bin)
        else:
            st.step("Tidak ada checkpoint step2_best di sesi mana pun")

    STEP2_SKIP_TRAINING = (not FORCE_RETRAIN_STEP2) and step2_already_done
    st.step("Keputusan: " + ("CACHE HIT → sel 8d-8e dilewati"
                             if STEP2_SKIP_TRAINING else
                             f"TRAINING dijalankan ({NUM_EPOCHS} epoch)"))

    if STEP2_SKIP_TRAINING:
        print(f"⏩ [CACHE HIT] Model Step 2 : {step2_ckpt}")
        if os.path.exists(step2_csv):
            df_s2_saved = pd.read_csv(step2_csv)
            step2_history = df_s2_saved.to_dict('records')
            _row_c2, best_step2_f1, best2_epoch = best_epoch_row(step2_history)
            best2_epoch = best2_epoch or NUM_EPOCHS
            _ada_hitungan = all(c in df_s2_saved.columns for c in ("tp", "fp", "fn"))
            st.step(f"Riwayat tersimpan: {len(df_s2_saved)} epoch, terbaik epoch {best2_epoch}"
                    + (" (TP/FP/FN tersedia)" if _ada_hitungan
                       else " (tanpa kolom TP/FP/FN — CSV dari run lama)"))
            if len(df_s2_saved) < NUM_EPOCHS:
                st.note(f"⚠️ Riwayat hanya {len(df_s2_saved)}/{NUM_EPOCHS} epoch — "
                        f"checkpoint dari run yang terhenti. Set FORCE_RETRAIN_STEP2=True "
                        f"bila ingin melatih penuh.")
        else:
            step2_history = []
            best_step2_f1 = 0.0
            best2_epoch = NUM_EPOCHS
            st.step("Riwayat CSV tidak ada — metrik per epoch tidak bisa dilaporkan")
# ── Resume Epoch: cek apakah training Step 2 terhenti di tengah jalan ────────
step2_resume_json = os.path.join(session_dirs["logs"], "step2_resume.json")
STEP2_RESUME_EPOCH = 0  # epoch terakhir yang selesai (0 = belum ada / baru)

if (not STEP2_SKIP_TRAINING) and (not FORCE_RETRAIN_STEP2):
    if not os.path.exists(step2_resume_json):
        _found_resume2 = auto_find_file("step2_resume.json", search_roots=[
            results_base if 'results_base' in globals() else "",
            "/content/drive/MyDrive/ACOS/Output/results",
            os.path.join(base_project_dir, "Output", "results"),
        ])
        if _found_resume2 and os.path.exists(_found_resume2):
            try:
                _rj2_prev = json.load(open(_found_resume2, encoding="utf-8"))
                _last2_prev = int(_rj2_prev.get("last_completed_epoch", 0))
                _prev_session_root2 = os.path.dirname(os.path.dirname(_found_resume2))
                _prev_epoch_dir2 = os.path.join(_prev_session_root2, "checkpoints", f"step2_epoch_{_last2_prev}")
                if os.path.isdir(_prev_epoch_dir2):
                    _tgt_epoch_dir2 = os.path.join(session_dirs["checkpoints"], f"step2_epoch_{_last2_prev}")
                    os.makedirs(_tgt_epoch_dir2, exist_ok=True)
                    for _f in os.listdir(_prev_epoch_dir2):
                        shutil.copy(os.path.join(_prev_epoch_dir2, _f), os.path.join(_tgt_epoch_dir2, _f))
                    shutil.copy(_found_resume2, step2_resume_json)
                    print(f"↪ [RESUME] Menyalin resume checkpoint Step 2 dari sesi lama: epoch {_last2_prev}")
            except Exception as _e_copy2:
                print(f"⚠️ Gagal menyalin resume Step 2 dari sesi lama: {_e_copy2}")

    if os.path.exists(step2_resume_json):
        try:
            _rj2 = json.load(open(step2_resume_json, encoding="utf-8"))
            _last2  = int(_rj2.get("last_completed_epoch", 0))
            _total2 = int(_rj2.get("total_epochs", NUM_EPOCHS))
            _epoch_ckpt_check2 = os.path.join(
                session_dirs["checkpoints"], f"step2_epoch_{_last2}")
            _epoch_bin_check2  = os.path.join(_epoch_ckpt_check2, "pytorch_model.bin")
            if _total2 != NUM_EPOCHS:
                print(f"⚠️  [RESUME] NUM_EPOCHS berubah ({_total2}→{NUM_EPOCHS}). "
                      f"Resume Step 2 diabaikan — training dari awal.")
            elif _last2 > 0 and _last2 < NUM_EPOCHS and os.path.exists(_epoch_bin_check2):
                STEP2_RESUME_EPOCH = _last2
                print(f"♻️  [RESUME] Step 2 terhenti di epoch {_last2}/{NUM_EPOCHS}. "
                      f"Akan dilanjutkan dari epoch {_last2 + 1}.")
            elif _last2 >= NUM_EPOCHS:
                print(f"✅ [RESUME] step2_resume.json menunjukkan training sudah "
                      f"selesai ({_last2}/{NUM_EPOCHS} epoch).")
        except Exception as _re2:
            print(f"⚠️  Tidak bisa membaca step2_resume.json: {_re2}. Training dari awal.")

elif STEP2_SKIP_TRAINING:
    if os.path.exists(step2_resume_json):
        try:
            _rj2_info = json.load(open(step2_resume_json, encoding="utf-8"))
            _ep2_info = _rj2_info.get("last_completed_epoch", "?")
            print(f"ℹ️  step2_resume.json ada (epoch {_ep2_info}/{NUM_EPOCHS}) "
                  f"— diabaikan karena cache hit sudah lengkap.")
        except Exception:
            pass

if STEP2_RESUME_EPOCH > 0:
    STEP2_SKIP_TRAINING = False  # paksa jalankan training (lanjut dari epoch berikutnya)
    print(f"   → STEP2_SKIP_TRAINING di-override: training akan lanjut dari "
          f"epoch {STEP2_RESUME_EPOCH + 1}.")

### 8c. Data Evaluasi Pasangan & Gold Step 2
Berbeda dari 5c: sel ini **tetap berjalan saat cache hit** kalau `eval_loader_2` belum ada
di memori, karena evaluasi final (9a) memerlukannya. Sumber pasangan dilaporkan eksplisit —
`_test_pair_1st.tsv` berarti skor pipeline penuh, `_test_pair.tsv` berarti Step 2 terisolasi.

In [ ]:
ensure_objects()
require_vars("step_stage", "processor_step2", "label_list_step2", "tokenizer")

_need_eval_loader = ("eval_loader_2" not in globals()) or ("eval_gold_2" not in globals())
if not _need_eval_loader:
    print("⏩ 8c dilewati — eval_loader_2 dan eval_gold_2 sudah ada di memori.")
else:
    with step_stage("8c. Data evaluasi pasangan + gold Step 2", 5) as st:
        tokenized_dir = os.path.join(tokenized_base, "tokenized_data")
        eval_pair_file, pakai_1st = resolve_eval_pair_file(tokenized_dir, DOMAIN,
                                                          prefer_1st=True)
        st.step(f"Sumber pasangan: {os.path.basename(eval_pair_file)} → "
                + ("prediksi step 1 (skor pipeline penuh)" if pakai_1st
                   else "gold pair (step 2 TERISOLASI, bukan skor pipeline)"))

        eval_examples_2 = pair_examples_from_file(processor_step2, eval_pair_file,
                                                 set_type="test")
        st.step(f"{len(eval_examples_2):,} contoh pasangan dibaca")

        eval_features_2 = features_step2(eval_examples_2, label_list_step2, MAX_SEQ_LENGTH,
                                         tokenizer, output_modes["categorysenti"])
        st.step(f"{len(eval_features_2):,} fitur dibentuk (max_seq_length={MAX_SEQ_LENGTH})")

        pin_mem = torch.cuda.is_available()
        num_work = 0 if sys.platform.startswith('win') else 2
        ev2_data = TensorDataset(
            torch.tensor([f.tokens_len for f in eval_features_2], dtype=torch.long),
            torch.tensor([f.aspect_input_ids for f in eval_features_2], dtype=torch.long),
            torch.tensor([f.aspect_input_mask for f in eval_features_2], dtype=torch.long),
            torch.tensor([f.aspect_segment_ids for f in eval_features_2], dtype=torch.long),
            torch.tensor([f.candidate_aspect for f in eval_features_2], dtype=torch.long),
            torch.tensor([f.candidate_opinion for f in eval_features_2], dtype=torch.long),
            torch.tensor([f.label_id for f in eval_features_2], dtype=torch.float)
        )
        eval_loader_2 = DataLoader(ev2_data, sampler=SequentialSampler(ev2_data),
                                   batch_size=16, pin_memory=pin_mem, num_workers=num_work)
        st.step(f"eval_loader_2 siap: {len(eval_loader_2)} batch × 16")

        class ArgsProxy:
            def __init__(self):
                self.bert_model = bert_cache_dir
                self.do_lower_case = True

        gold_pair_tsv = os.path.join(tokenized_dir, f"{DOMAIN}_test_pair.tsv")
        with open(gold_pair_tsv, "r", encoding="utf-8") as f:
            eval_gold_2 = read_pair_gold(f.readlines(), ArgsProxy())
        st.step(f"Gold quadruple dibaca dari {os.path.basename(gold_pair_tsv)} "
                f"({len(eval_gold_2):,} entri)")

### 8d. Model, Data Training & Optimizer Step 2

In [ ]:
require_vars("step_stage", "STEP2_SKIP_TRAINING")

if STEP2_SKIP_TRAINING:
    print("⏩ 8d dilewati — model dan optimizer tidak diperlukan saat cache hit.")
else:
    require_vars("eval_loader_2", "eval_gold_2", "num_labels_step2")
    with step_stage("8d. Model Category-Sentiment, data training, optimizer", 5) as st:
        if globals().get("USE_DUAL_HEAD", False):
            require_vars("num_labels_cat", "num_labels_senti")
            model_step2 = CategorySentiDualHead.from_pretrained(
                bert_cache_dir,
                num_categories=num_labels_cat,
                num_sentiments=num_labels_senti).to(device)
            st.step(f"Model: CategorySentiDualHead ({num_labels_cat} cat + {num_labels_senti} senti)")
        else:
            model_step2 = CategorySentiClassification.from_pretrained(
                bert_cache_dir, num_labels=num_labels_step2).to(device)
            st.step(f"Model: CategorySentiClassification ({num_labels_step2} label gabungan)")
        _n_par2 = sum(p.numel() for p in model_step2.parameters())
        _vram2 = torch.cuda.memory_allocated(device) / 1024 ** 2 if torch.cuda.is_available() else 0.0
        st.step(f"Model dimuat ke {device}: {_n_par2 / 1e6:.1f} M parameter, "
                f"VRAM terpakai {_vram2:.0f} MB")

        train_examples_2 = processor_step2.get_train_examples(tokenized_base, DOMAIN)
        st.step(f"{len(train_examples_2):,} contoh training dibaca")

        train_features_2 = features_step2(train_examples_2, label_list_step2, MAX_SEQ_LENGTH,
                                          tokenizer, output_modes["categorysenti"])
        tr2_data = TensorDataset(
            torch.tensor([f.tokens_len for f in train_features_2], dtype=torch.long),
            torch.tensor([f.aspect_input_ids for f in train_features_2], dtype=torch.long),
            torch.tensor([f.aspect_input_mask for f in train_features_2], dtype=torch.long),
            torch.tensor([f.aspect_segment_ids for f in train_features_2], dtype=torch.long),
            torch.tensor([f.candidate_aspect for f in train_features_2], dtype=torch.long),
            torch.tensor([f.candidate_opinion for f in train_features_2], dtype=torch.long),
            torch.tensor([f.label_id for f in train_features_2], dtype=torch.float)
        )
        train_loader_2 = DataLoader(tr2_data, sampler=RandomSampler(tr2_data),
                                    batch_size=STEP2_BATCH_SIZE, pin_memory=pin_mem,
                                    num_workers=num_work)
        st.step(f"train_loader_2 siap: {len(train_loader_2)} batch × {STEP2_BATCH_SIZE}")

        num_train_steps_2 = len(train_loader_2) * NUM_EPOCHS
        param_opt2 = list(model_step2.named_parameters())
        no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
        opt_grouped2 = [
            {'params': [p for n, p in param_opt2 if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
            {'params': [p for n, p in param_opt2 if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
        ]
        optimizer_2 = BertAdam(opt_grouped2, lr=STEP2_LR, warmup=0.1, t_total=num_train_steps_2)
        st.step(f"BertAdam siap: lr={STEP2_LR}, warmup=0.1, t_total={num_train_steps_2:,} step")
        st.step(f"logger2 & args_h dari sel 8a → {args_h.output_dir}")

### 8e. Loop Training Step 2
Bar epoch dengan ETA, bar batch dengan loss berjalan, satu baris ringkasan per epoch.
Progres per epoch ditulis ke `csv/step2_training_history.csv`, `logs/step2_progress.json`,
dan `session_manifest.json`.

In [ ]:
require_vars("step_stage", "STEP2_SKIP_TRAINING")

if STEP2_SKIP_TRAINING:
    print("⏩ 8e dilewati — training Step 2 tidak dijalankan (cache hit).")
    print(f"   Micro-F1 terbaik tersimpan: {best_step2_f1 * 100:.2f}% (epoch {best2_epoch})")
else:
    require_vars("model_step2", "optimizer_2", "train_loader_2", "eval_loader_2")
    with step_stage(f"8e. Training Step 2 Category-Sentiment — {NUM_EPOCHS} epoch pada {device}",
                    NUM_EPOCHS) as st:
        # ── Resume State ─────────────────────────────────────────────────────
        step2_resume_json = os.path.join(session_dirs["logs"], "step2_resume.json")
        start_epoch2   = 1
        best_step2_f1  = 0.0
        best2_epoch    = 1
        step2_history  = []
        epochs_since_best_2 = 0  # Counter untuk early stopping
        early_stopped_step2 = False  # Flag apakah training Step 2 berhenti karena early stopping

        _resume_ep2 = globals().get("STEP2_RESUME_EPOCH", 0)
        if _resume_ep2 > 0 and not FORCE_RETRAIN_STEP2:
            _epoch_ckpt_r2 = os.path.join(
                session_dirs["checkpoints"], f"step2_epoch_{_resume_ep2}")
            _model_path_r2 = os.path.join(_epoch_ckpt_r2, "pytorch_model.bin")
            _opt_path_r2   = os.path.join(_epoch_ckpt_r2, "optimizer.pt")
            try:
                model_step2.load_state_dict(
                    torch.load(_model_path_r2, map_location=device))
                st.note(f"✅ Bobot model Step 2 direstorasi dari epoch {_resume_ep2}")
                if os.path.exists(_opt_path_r2):
                    optimizer_2.load_state_dict(
                        torch.load(_opt_path_r2, map_location="cpu"))
                    st.note(f"✅ Optimizer state Step 2 direstorasi dari epoch {_resume_ep2}")
                else:
                    st.note(f"⚠️  optimizer.pt tidak ada — optimizer Step 2 mulai baru "
                            f"(bobot model tetap dari epoch {_resume_ep2})")
                _rj_r2 = json.load(open(step2_resume_json, encoding="utf-8"))
                step2_history = _rj_r2.get("history", [])
                best_step2_f1 = float(_rj_r2.get("best_micro_f1", 0.0))
                best2_epoch   = int(_rj_r2.get("best_epoch", 1))
                start_epoch2  = _resume_ep2 + 1
                st.step(f"♻️  Resume Step 2 dari epoch {_resume_ep2} → mulai epoch {start_epoch2} "
                        f"| best F1 sejauh ini: {best_step2_f1 * 100:.2f}% "
                        f"(epoch {best2_epoch})")
            except Exception as _load_err2:
                st.note(f"❌ Gagal load resume checkpoint Step 2: {_load_err2}. "
                        f"Training dimulai dari awal (epoch 1).")
                start_epoch2  = 1
                best_step2_f1 = 0.0
                best2_epoch   = 1
                step2_history = []
        else:
            st.step("Memulai training Step 2 baru dari epoch 1")
        # ─────────────────────────────────────────────────────────────────────

        _max_run2 = globals().get("MAX_EPOCHS_THIS_RUN", 0)
        _run_until2 = (start_epoch2 + _max_run2 - 1) if _max_run2 else NUM_EPOCHS
        _run_until2 = min(_run_until2, NUM_EPOCHS)
        _use_amp2 = globals().get("USE_AMP", True) and torch.cuda.is_available()
        scaler2 = torch.cuda.amp.GradScaler(enabled=_use_amp2)
        epoch_bar = tqdm(range(start_epoch2, _run_until2 + 1), desc="Step 2 epoch",
                         unit="epoch", initial=start_epoch2 - 1, total=_run_until2)
        for epoch in epoch_bar:
            model_step2.train()
            t_loss = 0.0
            batch_bar = tqdm(train_loader_2, desc=f"  epoch {epoch}/{NUM_EPOCHS}",
                             unit="batch", leave=False)
            for step, batch in enumerate(batch_bar, 1):
                batch = tuple(t.to(device) for t in batch)
                _len, _ids, _mask, _seg, _cand_a, _cand_o, _lbls = batch
                with torch.cuda.amp.autocast(enabled=_use_amp2):
                    out2 = model_step2(tokenizer, epoch, aspect_input_ids=_ids,
                                       aspect_token_type_ids=_seg, aspect_attention_mask=_mask,
                                       candidate_aspect=_cand_a, candidate_opinion=_cand_o,
                                       label_id=_lbls)
                loss, _ = unpack_model_output(out2)
                scaler2.scale(loss).backward()
                scaler2.step(optimizer_2)
                scaler2.update()
                optimizer_2.zero_grad(set_to_none=True)
                t_loss += loss.item()
                if step % 10 == 0 or step == len(train_loader_2):
                    batch_bar.set_postfix(loss=f"{t_loss / step:.4f}")
            batch_bar.close()

            avg_loss = t_loss / len(train_loader_2)
            model_step2.eval()
            print(f"   Epoch {epoch:02d}: evaluasi pasangan ({len(eval_loader_2)} batch)...",
                  flush=True)
            val_res = pair_eval(epoch, args_h, logger2, tokenizer, model_step2, eval_loader_2,
                                eval_gold_2, label_list_step2, device, "categorysenti",
                                eval_type='test')
            val_f1 = val_res.get('micro-F1', 0.0)
            # tp/fp/fn tersedia karena patch_eval_metrics_counts() di sel 8a.
            val_tp = float(val_res.get('tp', float('nan')))
            val_fp = float(val_res.get('fp', float('nan')))
            val_fn = float(val_res.get('fn', float('nan')))
            peak_vram2 = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if torch.cuda.is_available() else 0.0

            st.step(f"Epoch {epoch:02d} | loss {avg_loss:.4f} | TP {val_tp:.0f} FP {val_fp:.0f} "
                    f"FN {val_fn:.0f} | P {val_res.get('precision', 0.0) * 100:.2f}% "
                    f"| R {val_res.get('recall', 0.0) * 100:.2f}% "
                    f"| quadruple micro-F1 {val_f1 * 100:.2f}% | peak VRAM {peak_vram2:.0f} MB")

            _entry2 = {
                "epoch": epoch, "loss": avg_loss,
                "tp": val_tp, "fp": val_fp, "fn": val_fn,
                "precision": val_res.get('precision', 0.0),
                "recall": val_res.get('recall', 0.0),
                "micro-F1": val_f1,
                "peak_vram_mb": round(peak_vram2, 2)
            }

            if globals().get("USE_DUAL_HEAD", False):
                from sklearn.metrics import f1_score, accuracy_score
                _cp_list, _cg_list = [], []
                _sp_list, _sg_list = [], []
                with torch.no_grad():
                    for _eb in eval_loader_2:
                        _eb = tuple(t.to(device) for t in _eb)
                        model_step2(tokenizer, epoch, aspect_input_ids=_eb[1],
                                    aspect_token_type_ids=_eb[3], aspect_attention_mask=_eb[2],
                                    candidate_aspect=_eb[4], candidate_opinion=_eb[5],
                                    label_id=_eb[6])
                        _cl = getattr(model_step2, "latest_cat_logits", None)
                        _sl = getattr(model_step2, "latest_senti_logits", None)
                        if _cl is not None and _sl is not None:
                            _resh = _eb[6].view(-1, 13, 3)
                            _ct = _resh.sum(dim=-1).clamp(max=1.0)
                            _sp = _resh.sum(dim=1)
                            _has_s = (_sp.sum(dim=-1) > 0)
                            _cp_list.append((_cl.sigmoid() > 0.5).int().cpu().numpy())
                            _cg_list.append(_ct.int().cpu().numpy())
                            if _has_s.any():
                                _sp_list.append(_sl.argmax(dim=-1)[_has_s].cpu().numpy())
                                _sg_list.append(_sp.argmax(dim=-1)[_has_s].cpu().numpy())
                _val_cat_f1 = float(f1_score(np.vstack(_cg_list), np.vstack(_cp_list), average="micro", zero_division=0)) if _cp_list else 0.0
                _val_senti_acc = float(accuracy_score(np.concatenate(_sg_list), np.concatenate(_sp_list))) if _sp_list else 0.0
                _entry2["category_micro_f1"] = _val_cat_f1
                _entry2["sentiment_acc"] = _val_senti_acc
                st.step(f"   🎯 Dual-Head [Epoch {epoch:02d}]: Category micro-F1: {_val_cat_f1 * 100:.2f}% | Sentiment Acc: {_val_senti_acc * 100:.2f}%")

            step2_history.append(_entry2)

            if val_f1 > best_step2_f1 or epoch == 1 or not os.path.exists(step2_bin):
                if val_f1 > best_step2_f1:
                    best_step2_f1 = val_f1
                    best2_epoch = epoch
                    epochs_since_best_2 = 0  # Reset counter early stopping
                else:
                    epochs_since_best_2 += 1  # Increment counter early stopping
                torch.save(model_step2.state_dict(), step2_bin)
                model_step2.config.to_json_file(os.path.join(step2_ckpt, "config.json"))
                tokenizer.save_vocabulary(step2_ckpt)
                st.note(f"🔥 Checkpoint diperbarui (epoch {epoch}, F1 {val_f1 * 100:.2f}%) → {step2_ckpt}")
            else:
                epochs_since_best_2 += 1  # Increment counter early stopping

            # ── Rolling epoch checkpoint (resume per-epoch) ───────────────────
            _epoch_ckpt_dir2 = os.path.join(
                session_dirs["checkpoints"], f"step2_epoch_{epoch}")
            os.makedirs(_epoch_ckpt_dir2, exist_ok=True)
            torch.save(model_step2.state_dict(),
                       os.path.join(_epoch_ckpt_dir2, "pytorch_model.bin"))
            torch.save(optimizer_2.state_dict(),
                       os.path.join(_epoch_ckpt_dir2, "optimizer.pt"))
            st.note(f"💾 Rolling checkpoint Step 2 epoch {epoch} disimpan")

            # Hapus rolling checkpoint epoch sebelumnya (hemat storage)
            _prev_epoch_dir2 = os.path.join(
                session_dirs["checkpoints"], f"step2_epoch_{epoch - 1}")
            if epoch > start_epoch2 and os.path.isdir(_prev_epoch_dir2):
                shutil.rmtree(_prev_epoch_dir2, ignore_errors=True)
                st.note(f"🗑️  Rolling checkpoint Step 2 epoch {epoch - 1} dihapus")

            # ── Early stopping check ──────────────────────────────────────────
            if (PATIENCE > 0 and epoch >= MIN_EPOCHS_BEFORE_STOP and 
                epochs_since_best_2 >= PATIENCE):
                st.note(f"⏹️ Early stopping: F1 Step 2 tidak membaik selama {PATIENCE} epoch "
                        f"(terbaik di epoch {best2_epoch}, F1 {best_step2_f1 * 100:.2f}%)")
                early_stopped_step2 = True
                break
            # ─────────────────────────────────────────────────────────────────

            # Update resume JSON setiap akhir epoch
            with open(step2_resume_json, "w", encoding="utf-8") as _rjfw2:
                json.dump({
                    "last_completed_epoch": epoch,
                    "total_epochs": NUM_EPOCHS,
                    "best_micro_f1": best_step2_f1,
                    "best_epoch": best2_epoch,
                    "history": step2_history,
                    "early_stopped": early_stopped_step2,
                    "stopped_at_epoch": epoch if early_stopped_step2 else None,
                    "saved_at": datetime.now().isoformat(),
                }, _rjfw2, indent=2)
            # ─────────────────────────────────────────────────────────────────

            pd.DataFrame(step2_history).to_csv(step2_csv, index=False, encoding="utf-8")
            write_stage_progress(step2_progress_json, stage="STEP2_TRAINING", epoch=epoch,
                                 total_epochs=NUM_EPOCHS, last_loss=avg_loss,
                                 last_tp=val_tp, last_fp=val_fp, last_fn=val_fn,
                                 last_micro_f1=val_f1, best_micro_f1=best_step2_f1,
                                 best_epoch=best2_epoch,
                                 peak_vram_mb=round(peak_vram2, 2))
            update_mcp_manifest("STEP2_TRAINING", 5, {
                "step2_epoch_progress": f"{epoch}/{NUM_EPOCHS}",
                "step2_best_micro_f1": float(best_step2_f1 * 100),
                "step2_best_epoch": best2_epoch,
            })
            epoch_bar.set_postfix(best_f1=f"{best_step2_f1 * 100:.2f}%",
                                  loss=f"{avg_loss:.4f}")
        epoch_bar.close()

        # Hapus rolling checkpoint setelah training selesai penuh
        _last_rolling_dir2 = os.path.join(
            session_dirs["checkpoints"], f"step2_epoch_{NUM_EPOCHS}")
        if os.path.isdir(_last_rolling_dir2):
            shutil.rmtree(_last_rolling_dir2, ignore_errors=True)
            st.note(f"🗑️  Rolling checkpoint Step 2 epoch final dihapus (training selesai penuh)")

        if not os.path.exists(step2_bin):
            torch.save(model_step2.state_dict(), step2_bin)
            model_step2.config.to_json_file(os.path.join(step2_ckpt, "config.json"))
            tokenizer.save_vocabulary(step2_ckpt)
            st.note(f"💾 Checkpoint final Step 2 disimpan → {step2_ckpt}")

        if early_stopped_step2:
            print(f"⏹️ Training Step 2 berhenti (early stopping) di epoch {epoch}. "
                  f"Micro-F1 terbaik {best_step2_f1 * 100:.2f}% pada epoch {best2_epoch}.", 
                  flush=True)
        else:
            print(f"🏁 Training Step 2 selesai. Micro-F1 terbaik {best_step2_f1 * 100:.2f}% "
                  f"pada epoch {best2_epoch}.", flush=True)

# Ringkasan satu berkas untuk kedua cabang (training maupun cache hit).
step2_run_json = os.path.join(session_dirs["logs"], "step2_run_result.json")
_best_row2, _best_f1_2, _best_ep2 = best_epoch_row(globals().get("step2_history", []))
with open(step2_run_json, "w", encoding="utf-8") as _jf:
    json.dump({
        "mode": "cache_hit" if STEP2_SKIP_TRAINING else "trained",
        "domain": DOMAIN,
        "use_dual_head": globals().get("USE_DUAL_HEAD", False),
        "total_epochs_target": NUM_EPOCHS,
        "epochs_recorded": len(globals().get("step2_history", [])),
        "best_epoch": _best_ep2 or best2_epoch,
        "best_micro_f1": _best_f1_2,
        "best_micro_f1_pct": round(_best_f1_2 * 100, 2),
        "early_stopped": globals().get("early_stopped_step2", False),
        "stopped_at_epoch": epoch if globals().get("early_stopped_step2", False) else None,
        "best_row": _best_row2,
        "history": globals().get("step2_history", []),
        "checkpoint": step2_ckpt,
        "csv": step2_csv,
        "sumber_kandidat": "step1" if globals().get("pakai_1st", True) else "gold",
        "saved_at": datetime.now().isoformat(),
    }, _jf, indent=2)
print(f"🧾 Ringkasan run Step 2 (termasuk TP/FP/FN per epoch) → {step2_run_json}")

### 8f. Plot, Tabel & State Step 2